# Sacre Coeur: Gaussian splat training on Colab (Yushan's experiment)
Trains a new PhotoTourism scene with the team pipeline (`pipeline/reconstruction/run.py`, unchanged) using
`experiments/reconstruction/sacre-coeur/scene.json`.

- **Runtime:** Runtime → Change runtime type → GPU (T4 works; L4/A100 is faster).
- **Storage:** the 4.1 GB dataset is downloaded and extracted on the Colab disk (`/content/data`), never on a laptop.
  Only results (config, checkpoint, `building.ply`, metrics, previews) are copied to Google Drive.
- **Status:** this notebook has not been run yet. Nerfstudio/gsplat installation on Colab is unverified.

In [ ]:
!nvidia-smi
import sys; print(sys.version)

In [ ]:
# Public repo: no token needed.
BRANCH = "yushan/splat-experiment"
!git clone -q --branch {BRANCH} https://github.com/carldjapardi/Splatastrophe.git /content/Splatastrophe || (cd /content/Splatastrophe && git pull -q)
%cd /content/Splatastrophe
!git log --oneline -1

In [ ]:
# Same Nerfstudio version as the team install script. gsplat compiles its CUDA kernels on first use (several minutes).
!pip install -q 'nerfstudio==1.1.5' 'numpy<2' ninja
!ns-train splatfacto --help > /dev/null && echo "nerfstudio OK"

In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Splatastrophe')
SCENE = REPO / 'experiments/reconstruction/sacre-coeur/scene.json'
DATA_ROOT = Path('/content/data')          # temporary Colab disk for the 4 GB dataset
DRIVE_OUT = Path('/content/drive/MyDrive/splatastrophe/sacre-coeur')  # persistent results only
DATA_ROOT.mkdir(parents=True, exist_ok=True); DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def stage(name, *extra):
    cmd = [sys.executable, str(REPO / 'pipeline/reconstruction/run.py'), name,
           '--scene', str(SCENE), '--data-root', str(DATA_ROOT), *extra, '--execute']
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
# Download (~4.1 GB from UBC) + NeRF-W split, convert supplied COLMAP cameras, make splits, inspect cameras.
# Run once per Colab session; the split stage refuses to run twice on the same processed folder.
stage('download')
stage('process')
stage('split')
stage('inspect')

In [ ]:
# 1,000-iteration smoke test: confirms gsplat compiles and the data loads before the long run.
stage('train', '--iterations', '1000')

In [ ]:
# Full run: 30,000 iterations (Splatfacto default). Expect roughly 30-90 min depending on the GPU.
stage('train', '--iterations', '30000')

In [ ]:
# Evaluate, export building.ply, render validation previews for the newest run.
runs = sorted((DATA_ROOT / 'outputs/sacre-coeur').rglob('config.yml'), key=lambda p: p.stat().st_mtime)
CONFIG = runs[-1]; print('Using', CONFIG)
stage('eval', '--load-config', str(CONFIG))
stage('export', '--load-config', str(CONFIG))
stage('preview', '--load-config', str(CONFIG))

In [ ]:
# Copy results (not the raw dataset) to Google Drive before the Colab session ends.
import shutil, json
run_dir = CONFIG.parent
dest = DRIVE_OUT / run_dir.name
shutil.copytree(run_dir, dest, dirs_exist_ok=True)
for extra in ['source.json']:
    src = DATA_ROOT / 'data/raw/phototourism/sacre-coeur' / extra
    if src.exists(): shutil.copy(src, dest / extra)
print('Saved to', dest)
m = run_dir / 'metrics.json'
if m.exists(): print(json.dumps(json.loads(m.read_text()).get('results', {}), indent=2))